# Chapter 7.8: Autonomous Recommendation Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. Design self-evolving recommendation systems with continuous learning
2. Implement automated feature engineering for recommendation
3. Build auto-architecture search for rec models
4. Understand Meta's and Google's auto-optimization approaches
5. Implement monitoring and safeguards for autonomous systems
6. Navigate the spectrum from human-in-the-loop to fully autonomous
7. Build a simple auto-evolving rec system with online adaptation

## Prerequisites

- Neural network fundamentals (PyTorch)
- Understanding of hyperparameter optimization
- Chapters 7.1-7.7 (RL and agent concepts)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part7/chapter_7.8_autonomous.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part7/chapter_7.8_autonomous.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, deque
from dataclasses import dataclass, field
from copy import deepcopy

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. The Spectrum of Autonomy

Recommendation systems operate on a spectrum of autonomy:

| Level | Description | Example |
|-------|------------|--------|
| 0 | Manual rules | Hand-curated editorial picks |
| 1 | Supervised learning | Trained model, human-selected features |
| 2 | Auto-tuned | Automated hyperparameter optimization |
| 3 | Auto-feature | Automated feature engineering |
| 4 | Auto-architecture | Neural architecture search for rec |
| 5 | Fully autonomous | Self-evolving, self-monitoring |

> **💡 Concept:** Meta's recommendation systems use automated tools to optimize models with minimal human intervention (Naumov et al., 2019 — DLRM). Google's AutoML-Zero (Real et al., 2020) explores fully automatic ML pipeline discovery. The trend is toward greater autonomy, but with robust monitoring and safeguards.

### Key Challenges

1. **Distribution shift**: User behavior changes over time
2. **Concept drift**: What "good" means evolves
3. **Safety**: Autonomous systems can amplify biases
4. **Observability**: Must detect when the system goes wrong

In [ ]:
# Simulated streaming recommendation environment
class StreamingRecEnv:
    """Non-stationary recommendation environment that evolves over time."""
    
    def __init__(self, n_users: int = 100, n_items: int = 50,
                 feature_dim: int = 10, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.n_users = n_users
        self.n_items = n_items
        self.feature_dim = feature_dim
        
        # Raw features
        self.user_features = self.rng.randn(n_users, feature_dim) * 0.5
        self.item_features = self.rng.randn(n_items, feature_dim) * 0.5
        
        # True interaction weights (unknown to the model)
        self.true_weights = self.rng.randn(feature_dim, feature_dim) * 0.3
        
        self.timestep = 0
        self.drift_rate = 0.002  # How fast preferences change
    
    def get_features(self, user_id: int, item_id: int) -> np.ndarray:
        """Get feature vector for a user-item pair."""
        u = self.user_features[user_id]
        i = self.item_features[item_id]
        # Interaction features
        cross = np.outer(u, i).flatten()[:self.feature_dim]  # Simplified
        return np.concatenate([u, i, cross, u * i]).astype(np.float32)
    
    def get_reward(self, user_id: int, item_id: int) -> float:
        """Get reward (click/no-click)."""
        u = self.user_features[user_id]
        i = self.item_features[item_id]
        logit = u @ self.true_weights @ i
        prob = 1.0 / (1.0 + np.exp(-logit))
        return float(self.rng.random() < prob)
    
    def step_time(self):
        """Advance time: introduce distribution shift."""
        self.timestep += 1
        
        # Gradual preference drift
        drift = self.rng.randn(*self.true_weights.shape) * self.drift_rate
        self.true_weights += drift
        
        # Occasional sudden shift (every 200 steps)
        if self.timestep % 200 == 0:
            self.true_weights += self.rng.randn(*self.true_weights.shape) * 0.05
            print(f"  [Step {self.timestep}] Sudden distribution shift!")
        
        # New items appear
        if self.timestep % 50 == 0:
            idx = self.rng.randint(self.n_items)
            self.item_features[idx] = self.rng.randn(self.feature_dim) * 0.5


env = StreamingRecEnv(seed=42)
feat_dim = len(env.get_features(0, 0))
print(f"Environment: {env.n_users} users, {env.n_items} items, feature dim: {feat_dim}")

## 2. Online Learning: Continuous Model Updates

The simplest form of autonomy is **online learning** — updating the model continuously as new data arrives.

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(f_\theta(x_t), y_t)$$

Challenges:
- **Catastrophic forgetting**: Old patterns may be overwritten
- **Learning rate scheduling**: Too fast = unstable, too slow = can't adapt
- **Data recency weighting**: Recent data should matter more

In [ ]:
class OnlineRecModel(nn.Module):
    """Simple online-learning recommendation model."""
    
    def __init__(self, input_dim: int, hidden_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class OnlineLearner:
    """Online learning wrapper with replay buffer."""
    
    def __init__(self, model: nn.Module, lr: float = 1e-3,
                 buffer_size: int = 1000, batch_size: int = 32):
        self.model = model
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.buffer = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        self.update_count = 0
    
    def predict(self, features: np.ndarray) -> float:
        with torch.no_grad():
            x = torch.FloatTensor(features)
            return float(self.model(x))
    
    def update(self, features: np.ndarray, reward: float):
        """Add to buffer and train."""
        self.buffer.append((features, reward))
        
        if len(self.buffer) >= self.batch_size:
            batch = random.sample(list(self.buffer), self.batch_size)
            features_batch = torch.FloatTensor(np.array([b[0] for b in batch]))
            rewards_batch = torch.FloatTensor([b[1] for b in batch])
            
            preds = self.model(features_batch)
            loss = F.binary_cross_entropy(preds, rewards_batch)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.update_count += 1


class StaticModel:
    """Baseline: model trained once and never updated."""
    
    def __init__(self, input_dim: int):
        self.model = OnlineRecModel(input_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-3)
        self.trained = False
    
    def initial_train(self, data: List[Tuple], n_epochs: int = 10):
        """Train on initial batch of data."""
        for _ in range(n_epochs):
            batch = random.sample(data, min(64, len(data)))
            features = torch.FloatTensor(np.array([b[0] for b in batch]))
            rewards = torch.FloatTensor([b[1] for b in batch])
            
            preds = self.model(features)
            loss = F.binary_cross_entropy(preds, rewards)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        self.trained = True
    
    def predict(self, features: np.ndarray) -> float:
        with torch.no_grad():
            return float(self.model(torch.FloatTensor(features)))


# Compare online vs static
env = StreamingRecEnv(seed=42)

online_model = OnlineLearner(OnlineRecModel(feat_dim), lr=1e-3)
static_baseline = StaticModel(feat_dim)

# Warm-up: collect initial data
warmup_data = []
for _ in range(500):
    u = np.random.randint(env.n_users)
    i = np.random.randint(env.n_items)
    feats = env.get_features(u, i)
    reward = env.get_reward(u, i)
    warmup_data.append((feats, reward))

# Train static model
static_baseline.initial_train(warmup_data, n_epochs=20)

# Warm up online model too
for feats, reward in warmup_data:
    online_model.update(feats, reward)

# Run streaming evaluation
n_steps = 500
online_rewards = []
static_rewards = []
window = 50

for step in range(n_steps):
    env.step_time()
    
    u = np.random.randint(env.n_users)
    
    # Online model recommendation
    best_score = -1
    best_item = 0
    for i in range(env.n_items):
        feats = env.get_features(u, i)
        score = online_model.predict(feats)
        if score > best_score:
            best_score = score
            best_item = i
    
    online_reward = env.get_reward(u, best_item)
    online_model.update(env.get_features(u, best_item), online_reward)
    online_rewards.append(online_reward)
    
    # Static model recommendation
    best_score = -1
    best_item = 0
    for i in range(env.n_items):
        feats = env.get_features(u, i)
        score = static_baseline.predict(feats)
        if score > best_score:
            best_score = score
            best_item = i
    
    static_reward = env.get_reward(u, best_item)
    static_rewards.append(static_reward)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 5))
online_smooth = np.convolve(online_rewards, np.ones(window)/window, mode='valid')
static_smooth = np.convolve(static_rewards, np.ones(window)/window, mode='valid')
ax.plot(online_smooth, label='Online Learning', color='steelblue')
ax.plot(static_smooth, label='Static Model', color='coral')
ax.axvline(x=150, color='gray', linestyle='--', alpha=0.5, label='Distribution shift')
ax.axvline(x=350, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('Online Learning vs Static Model Under Distribution Shift')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Online avg reward: {np.mean(online_rewards):.3f}")
print(f"Static avg reward: {np.mean(static_rewards):.3f}")

## 3. Automated Feature Engineering

Autonomous systems can discover useful features automatically by:

1. **Feature crossing**: Automatically combine features (e.g., user_age x item_category)
2. **Feature selection**: Drop uninformative features
3. **Feature transformation**: Apply log, sqrt, binning automatically

> **💡 Concept:** Google's Wide & Deep model (Cheng et al., 2016) automates feature crossing via the deep component. AutoFIS (Liu et al., 2020) uses architecture search to find optimal feature interactions.

In [ ]:
class AutoFeatureEngineer:
    """Automated feature engineering for recommendation."""
    
    def __init__(self, base_feature_dim: int):
        self.base_dim = base_feature_dim
        self.transformations = [
            ('identity', lambda x: x),
            ('square', lambda x: x ** 2),
            ('abs', lambda x: np.abs(x)),
            ('sign', lambda x: np.sign(x)),
            ('log1p', lambda x: np.log1p(np.abs(x)) * np.sign(x)),
        ]
        self.selected_transforms = list(range(len(self.transformations)))
        self.feature_importances = None
    
    def generate_features(self, base_features: np.ndarray) -> np.ndarray:
        """Generate engineered features."""
        all_features = [base_features]
        
        for idx in self.selected_transforms:
            name, transform = self.transformations[idx]
            try:
                transformed = transform(base_features)
                all_features.append(transformed)
            except (ValueError, RuntimeWarning):
                pass
        
        # Pairwise interactions (top features only to keep dimension manageable)
        top_k = min(5, len(base_features))
        for i in range(top_k):
            for j in range(i + 1, top_k):
                all_features.append(np.array([base_features[i] * base_features[j]]))
        
        return np.concatenate(all_features).astype(np.float32)
    
    def evaluate_features(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        """Evaluate feature importance using correlation with target."""
        importances = []
        for j in range(X.shape[1]):
            col = X[:, j]
            if np.std(col) < 1e-8:
                importances.append(0.0)
            else:
                corr = np.abs(np.corrcoef(col, y)[0, 1])
                importances.append(corr if not np.isnan(corr) else 0.0)
        
        self.feature_importances = np.array(importances)
        return self.feature_importances
    
    def select_top_features(self, threshold: float = 0.05) -> np.ndarray:
        """Select features above importance threshold."""
        if self.feature_importances is None:
            return np.arange(self.base_dim)
        return np.where(self.feature_importances > threshold)[0]


# Demo auto feature engineering
auto_fe = AutoFeatureEngineer(feat_dim)

# Generate features for some samples
X_samples = []
y_samples = []
for _ in range(500):
    u = np.random.randint(env.n_users)
    i = np.random.randint(env.n_items)
    base_feats = env.get_features(u, i)
    engineered = auto_fe.generate_features(base_feats)
    X_samples.append(engineered)
    y_samples.append(env.get_reward(u, i))

X = np.array(X_samples)
y = np.array(y_samples)

# Evaluate and select
importances = auto_fe.evaluate_features(X, y)
selected = auto_fe.select_top_features(threshold=0.03)

print(f"Base features: {feat_dim}")
print(f"Engineered features: {X.shape[1]}")
print(f"Selected features (importance > 0.03): {len(selected)}")

# Plot feature importances
fig, ax = plt.subplots(figsize=(12, 4))
sorted_idx = np.argsort(importances)[::-1][:30]  # Top 30
ax.bar(range(len(sorted_idx)), importances[sorted_idx], color='steelblue')
ax.axhline(y=0.03, color='red', linestyle='--', label='Selection threshold')
ax.set_xlabel('Feature Index (sorted)')
ax.set_ylabel('Importance (|correlation|)')
ax.set_title('Auto Feature Engineering: Feature Importances')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Auto-Architecture Search for Rec Models

Instead of hand-designing the model architecture, we can search over architectures.

Search space for rec models:
- Number of layers (1-5)
- Hidden dimensions (16, 32, 64, 128)
- Activation functions (ReLU, GELU, Tanh)
- Dropout rates (0, 0.1, 0.2, 0.3)
- Feature interaction types (concat, element-wise product, attention)

> **⚠️ Common Pitfall:** NAS is computationally expensive. For practical rec systems, constrain the search space and use efficient methods like random search or successive halving.

In [ ]:
class RecModelConfig:
    """Configuration for a recommendation model."""
    
    def __init__(self, n_layers: int = 2, hidden_dim: int = 32,
                 activation: str = 'relu', dropout: float = 0.0,
                 use_batch_norm: bool = False):
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        self.activation = activation
        self.dropout = dropout
        self.use_batch_norm = use_batch_norm
    
    def __repr__(self):
        return (f"Config(layers={self.n_layers}, hidden={self.hidden_dim}, "
                f"act={self.activation}, drop={self.dropout}, bn={self.use_batch_norm})")


def build_model_from_config(input_dim: int, config: RecModelConfig) -> nn.Module:
    """Build a PyTorch model from config."""
    activation_map = {
        'relu': nn.ReLU,
        'gelu': nn.GELU,
        'tanh': nn.Tanh,
    }
    
    layers = []
    prev_dim = input_dim
    
    for i in range(config.n_layers):
        layers.append(nn.Linear(prev_dim, config.hidden_dim))
        if config.use_batch_norm:
            layers.append(nn.BatchNorm1d(config.hidden_dim))
        layers.append(activation_map[config.activation]())
        if config.dropout > 0:
            layers.append(nn.Dropout(config.dropout))
        prev_dim = config.hidden_dim
    
    layers.append(nn.Linear(prev_dim, 1))
    layers.append(nn.Sigmoid())
    
    model = nn.Sequential(*layers)
    return model


class SimpleArchitectureSearch:
    """Random search over model architectures."""
    
    def __init__(self, input_dim: int, search_budget: int = 10):
        self.input_dim = input_dim
        self.search_budget = search_budget
        self.results = []
    
    def sample_config(self) -> RecModelConfig:
        return RecModelConfig(
            n_layers=random.choice([1, 2, 3, 4]),
            hidden_dim=random.choice([16, 32, 64, 128]),
            activation=random.choice(['relu', 'gelu', 'tanh']),
            dropout=random.choice([0.0, 0.1, 0.2, 0.3]),
            use_batch_norm=random.choice([True, False])
        )
    
    def evaluate_config(self, config: RecModelConfig, 
                        train_data: List[Tuple], val_data: List[Tuple],
                        n_epochs: int = 10) -> float:
        """Train and evaluate a model configuration."""
        model = build_model_from_config(self.input_dim, config)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        
        # Train
        model.train()
        for epoch in range(n_epochs):
            batch = random.sample(train_data, min(64, len(train_data)))
            X = torch.FloatTensor(np.array([b[0] for b in batch]))
            y = torch.FloatTensor([b[1] for b in batch])
            
            preds = model(X).squeeze(-1)
            loss = F.binary_cross_entropy(preds, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            X_val = torch.FloatTensor(np.array([b[0] for b in val_data]))
            y_val = torch.FloatTensor([b[1] for b in val_data])
            preds = model(X_val).squeeze(-1)
            val_loss = F.binary_cross_entropy(preds, y_val).item()
        
        n_params = sum(p.numel() for p in model.parameters())
        return val_loss, n_params
    
    def search(self, train_data: List[Tuple], val_data: List[Tuple]) -> RecModelConfig:
        """Search for the best architecture."""
        self.results = []
        
        for trial in range(self.search_budget):
            config = self.sample_config()
            val_loss, n_params = self.evaluate_config(config, train_data, val_data)
            self.results.append({
                'config': config,
                'val_loss': val_loss,
                'n_params': n_params
            })
            print(f"Trial {trial+1}: {config} -> loss={val_loss:.4f}, params={n_params}")
        
        best = min(self.results, key=lambda x: x['val_loss'])
        return best['config']


# Run architecture search
nas = SimpleArchitectureSearch(feat_dim, search_budget=12)

# Split data
train_data = warmup_data[:400]
val_data = warmup_data[400:]

best_config = nas.search(train_data, val_data)
print(f"\nBest config: {best_config}")

# Plot search results
fig, ax = plt.subplots(figsize=(10, 5))
losses = [r['val_loss'] for r in nas.results]
params = [r['n_params'] for r in nas.results]
ax.scatter(params, losses, s=100, c=range(len(losses)), cmap='viridis', zorder=5)
best_idx = np.argmin(losses)
ax.scatter(params[best_idx], losses[best_idx], s=200, c='red', marker='*', 
           zorder=10, label=f'Best (loss={losses[best_idx]:.4f})')
ax.set_xlabel('Number of Parameters')
ax.set_ylabel('Validation Loss')
ax.set_title('Architecture Search: Loss vs Model Size')
ax.legend()
plt.colorbar(ax.collections[0], label='Trial number')
plt.tight_layout()
plt.show()

## 5. Monitoring and Safeguards

Autonomous systems need robust monitoring to detect:

1. **Performance degradation**: CTR drops, latency increases
2. **Distribution shift**: Input features change distribution
3. **Fairness violations**: Bias amplification
4. **Safety issues**: Inappropriate content, manipulation

> **🔑 Pro Tip:** Always have automated rollback mechanisms. If a metric drops below a threshold, the system should automatically revert to the previous stable version.

In [ ]:
class SystemMonitor:
    """Monitor autonomous recommendation system health."""
    
    def __init__(self, window_size: int = 50, alert_threshold: float = 0.2):
        self.window_size = window_size
        self.alert_threshold = alert_threshold
        self.metrics_history = defaultdict(list)
        self.alerts: List[Dict] = []
        self.baseline_metrics = {}
    
    def set_baseline(self, metrics: Dict[str, float]):
        """Set baseline performance metrics."""
        self.baseline_metrics = metrics.copy()
    
    def record(self, step: int, metrics: Dict[str, float]):
        """Record metrics and check for anomalies."""
        for key, value in metrics.items():
            self.metrics_history[key].append(value)
        
        # Check for alerts
        for key in metrics:
            if key in self.baseline_metrics:
                recent = self.metrics_history[key][-self.window_size:]
                if len(recent) >= self.window_size:
                    current_avg = np.mean(recent)
                    baseline = self.baseline_metrics[key]
                    
                    # Check for significant degradation
                    if baseline > 0:
                        drop = (baseline - current_avg) / baseline
                        if drop > self.alert_threshold:
                            self.alerts.append({
                                'step': step,
                                'metric': key,
                                'baseline': baseline,
                                'current': current_avg,
                                'drop_pct': drop * 100
                            })
    
    def detect_distribution_shift(self, recent_features: np.ndarray,
                                    baseline_features: np.ndarray) -> float:
        """Detect distribution shift using simple statistics."""
        recent_mean = np.mean(recent_features, axis=0)
        baseline_mean = np.mean(baseline_features, axis=0)
        
        # Normalized mean difference
        baseline_std = np.std(baseline_features, axis=0) + 1e-8
        shift_score = np.mean(np.abs(recent_mean - baseline_mean) / baseline_std)
        return float(shift_score)
    
    def should_rollback(self) -> bool:
        """Decide if system should rollback to previous version."""
        if not self.alerts:
            return False
        # Rollback if multiple alerts in short period
        recent_alerts = [a for a in self.alerts if a['step'] >= 
                        max(0, len(self.metrics_history['ctr']) - self.window_size)]
        return len(recent_alerts) >= 2
    
    def get_status_report(self) -> str:
        """Generate a status report."""
        report = "System Status Report\n" + "=" * 40 + "\n"
        
        for key, values in self.metrics_history.items():
            if values:
                recent = values[-self.window_size:]
                report += (f"  {key}: current={np.mean(recent):.4f}, "
                          f"baseline={self.baseline_metrics.get(key, 'N/A')}\n")
        
        if self.alerts:
            report += f"\nAlerts ({len(self.alerts)}): \n"
            for a in self.alerts[-3:]:
                report += f"  Step {a['step']}: {a['metric']} dropped {a['drop_pct']:.1f}%\n"
        
        report += f"\nRollback recommended: {self.should_rollback()}\n"
        return report


# Run autonomous system with monitoring
env = StreamingRecEnv(seed=42)
model = OnlineLearner(OnlineRecModel(feat_dim), lr=1e-3)
monitor = SystemMonitor(window_size=30, alert_threshold=0.15)

# Warm up
warmup_rewards = []
for _ in range(100):
    u = np.random.randint(env.n_users)
    i = np.random.randint(env.n_items)
    feats = env.get_features(u, i)
    reward = env.get_reward(u, i)
    model.update(feats, reward)
    warmup_rewards.append(reward)

monitor.set_baseline({'ctr': np.mean(warmup_rewards)})

# Run with monitoring
n_steps = 400
monitored_rewards = []

for step in range(n_steps):
    env.step_time()
    u = np.random.randint(env.n_users)
    
    # Recommend
    best_score = -1
    best_item = 0
    for i in range(env.n_items):
        score = model.predict(env.get_features(u, i))
        if score > best_score:
            best_score = score
            best_item = i
    
    reward = env.get_reward(u, best_item)
    model.update(env.get_features(u, best_item), reward)
    monitored_rewards.append(reward)
    
    # Monitor
    monitor.record(step, {'ctr': reward})
    
    # Check for rollback
    if monitor.should_rollback():
        pass  # In production, would revert to previous model

print(monitor.get_status_report())

# Plot monitoring dashboard
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# CTR over time
window = 20
smoothed = np.convolve(monitored_rewards, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed, color='steelblue')
baseline = monitor.baseline_metrics['ctr']
axes[0].axhline(y=baseline, color='green', linestyle='--', label=f'Baseline ({baseline:.3f})')
axes[0].axhline(y=baseline * (1 - monitor.alert_threshold), color='red', linestyle='--',
               label=f'Alert threshold ({baseline * (1-monitor.alert_threshold):.3f})')

# Mark alerts
for alert in monitor.alerts:
    axes[0].axvline(x=alert['step'], color='red', alpha=0.3)

axes[0].set_ylabel('CTR (smoothed)')
axes[0].set_title('Autonomous System Monitoring')
axes[0].legend()

# Alert timeline
if monitor.alerts:
    alert_steps = [a['step'] for a in monitor.alerts]
    alert_drops = [a['drop_pct'] for a in monitor.alerts]
    axes[1].stem(alert_steps, alert_drops, linefmt='r-', markerfmt='ro', basefmt=' ')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Performance Drop (%)')
    axes[1].set_title('Alert Timeline')
else:
    axes[1].text(0.5, 0.5, 'No alerts triggered', ha='center', va='center',
                transform=axes[1].transAxes, fontsize=14)
    axes[1].set_title('Alert Timeline')

plt.tight_layout()
plt.show()

## 6. Summary

| Autonomy Level | What's Automated | Human Role | Risk Level |
|---------------|-----------------|------------|------------|
| Online Learning | Model weights | Monitor performance | Low |
| Auto-Feature | Feature engineering | Review features | Medium |
| Auto-Architecture | Model structure | Set search space | Medium |
| Full Autonomy | Everything | Set guardrails | High |

**Key references:**
- Naumov et al. (2019) - DLRM: Deep Learning Recommendation Model (Meta)
- Cheng et al. (2016) - Wide & Deep Learning for Recommender Systems (Google)
- Liu et al. (2020) - AutoFIS: Feature Interaction Selection (Huawei)
- Real et al. (2020) - AutoML-Zero (Google)
- Zhao et al. (2021) - AutoEmb: Automated embedding for rec

---

## Exercises

### 🏋️ Exercise 1: Build an Auto-Evolving Rec System

In [ ]:
# 🏋️ Exercise 1: Auto-evolving recommendation system

class AutoEvolvingRecSystem:
    """Fully autonomous recommendation system."""
    
    def __init__(self, input_dim: int):
        self.input_dim = input_dim
        self.current_model = None
        self.feature_engineer = None
        self.monitor = SystemMonitor()
        self.version = 0
    
    def initialize(self, warmup_data: List[Tuple]):
        # TODO: 
        # 1. Auto feature engineering
        # 2. Architecture search
        # 3. Train initial model
        # 4. Set monitoring baselines
        pass
    
    def recommend(self, user_features: np.ndarray, 
                  item_features: List[np.ndarray]) -> int:
        # TODO: Score items and return best
        pass
    
    def update(self, features: np.ndarray, reward: float):
        # TODO: Online update + monitoring
        pass
    
    def periodic_check(self):
        # TODO: 
        # 1. Check monitor alerts
        # 2. If degradation detected, re-run auto feature engineering
        # 3. If severe, re-run architecture search
        # 4. If critical, rollback
        pass

# TODO: Run for 1000 steps with distribution shifts
# Compare against static model and simple online model

print("Exercise 1: Implement AutoEvolvingRecSystem")

### 🏋️ Exercise 2: Implement Successive Halving for NAS

In [ ]:
# 🏋️ Exercise 2: Efficient architecture search

class SuccessiveHalvingSearch:
    """Successive halving for efficient NAS."""
    
    def __init__(self, input_dim: int, n_configs: int = 16, 
                 max_epochs: int = 20, halving_rate: int = 2):
        self.input_dim = input_dim
        self.n_configs = n_configs
        self.max_epochs = max_epochs
        self.halving_rate = halving_rate
    
    def search(self, train_data, val_data) -> RecModelConfig:
        # TODO:
        # 1. Sample n_configs random configurations
        # 2. Train each for 1 epoch, evaluate
        # 3. Keep top half, train for 2 more epochs
        # 4. Keep top half, train for 4 more epochs
        # 5. Continue until 1 config remains
        # Much more efficient than training all configs fully!
        pass

print("Exercise 2: Implement SuccessiveHalvingSearch")

### 🏋️ Exercise 3: Anomaly Detection for Model Monitoring

In [ ]:
# 🏋️ Exercise 3: Advanced anomaly detection

class AnomalyDetector:
    """Detect anomalies in recommendation system metrics."""
    
    def __init__(self, window_size: int = 50, n_sigma: float = 2.0):
        self.window_size = window_size
        self.n_sigma = n_sigma
        self.history = defaultdict(list)
    
    def detect(self, metrics: Dict[str, float]) -> List[Dict]:
        # TODO: Implement anomaly detection:
        # 1. For each metric, compute rolling mean and std
        # 2. Flag if current value is > n_sigma standard deviations away
        # 3. Also detect sudden changes (derivative-based)
        # 4. Return list of anomalies with severity
        pass
    
    def detect_trend_change(self, metric_name: str) -> bool:
        # TODO: Detect if the trend direction has changed
        # (e.g., metric was increasing but now decreasing)
        pass

print("Exercise 3: Implement AnomalyDetector")